# Qubits and Measurement
Chapter 4 companion. Predict, calculate by hand, run and explain. These are ideal simulations of known preparations, not hardware measurements or a method for reading all amplitudes from an unknown qubit.


In [ ]:
%pip -q install "qiskit>=2,<3" "numpy>=2,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit.quantum_info import Statevector
print("Qiskit", qiskit.__version__, "NumPy", np.__version__)
rng = np.random.default_rng(2026)
zero = np.array([1,0], dtype=complex)
one = np.array([0,1], dtype=complex)
plus = np.array([1,1], dtype=complex)/np.sqrt(2)
minus = np.array([1,-1], dtype=complex)/np.sqrt(2)
Z_basis = np.eye(2, dtype=complex)
X_basis = np.column_stack([plus,minus])
def normalize(v):
    v = np.asarray(v,dtype=complex)
    length = np.linalg.norm(v)
    if length == 0:
        raise ValueError("Zero cannot be normalized")
    return v/length
def probabilities(v,basis):
    assert np.isclose(np.vdot(v,v),1)
    return np.abs(basis.conj().T @ v)**2


## 4.1 Normalize a preparation
A constructor can store an unnormalized column; check validity explicitly. The column (1,i) is not a normalized qubit state until it is scaled.


In [ ]:
raw = np.array([1,1j])
assert not Statevector(raw).is_valid()
prepared = normalize(raw)
assert Statevector(prepared).is_valid()
print("Normalized (1,i):", prepared)
psi = normalize([1-1j,2])
assert np.allclose(probabilities(psi,Z_basis),[1/3,2/3])
try:
    normalize([0,0])
except ValueError as e:
    print(e)
else:
    raise AssertionError("Zero normalization must fail")


## 4.2 Exact Born probabilities and fresh-shot samples
`sample_counts` samples the current state probabilities without updating the stored state. Use it here to model repeated identical fresh preparations. It does **not** simulate retaining one collapsed qubit for all those readouts.


In [ ]:
psi = np.array([(1+2j)/3,-2/3])
sv = Statevector(psi)
assert sv.is_valid()
pz = probabilities(psi,Z_basis)
assert np.allclose(pz,[5/9,4/9])
assert np.allclose(sv.probabilities(),pz)
sv.seed(2026)
counts = sv.sample_counts(90)
assert sum(counts.values()) == 90
assert np.allclose(sv.data,psi)  # sampling has not changed the stored model
print("Exact:",pz,"Expected counts:",90*pz,"Observed:",counts)
fig,axes = plt.subplots(1,2,figsize=(8,3))
axes[0].bar(["0","1"],pz,color="#286a9b")
axes[0].set(title="Exact probabilities",ylim=(0,1))
axes[1].bar(["0","1"],[counts.get("0",0),counts.get("1",0)],color="#637d90")
axes[1].set(title="One sampled batch",ylabel="Count",ylim=(0,90))
fig.tight_layout()
plt.show()


## 4.3 Phase and a second basis
Columns of X_basis are the orthonormal X measurement vectors. Multiplying by its adjoint computes coordinates in that basis. Qiskit bit labels 0/1 on that coordinate array correspond to X labels +/− here. This is a mathematical basis calculation; gate implementation follows in Chapter 5.


In [ ]:
px = probabilities(psi,X_basis)
assert np.allclose(px,[5/18,13/18])
x_coordinates = X_basis.conj().T @ psi
assert np.allclose(Statevector(x_coordinates).probabilities(),px)
assert Statevector(1j*psi).equiv(sv)
for basis in [Z_basis,X_basis]:
    assert np.allclose(probabilities(1j*psi,basis), probabilities(psi,basis))
for phase,expected in [(0,1),(np.pi/2,.5),(np.pi,0)]:
    state = np.array([1,np.exp(1j*phase)])/np.sqrt(2)
    assert np.allclose(probabilities(state,Z_basis),[.5,.5])
    assert np.isclose(probabilities(state,X_basis)[0],expected)
    print("Relative phase",phase,"PX(+)",expected)


## 4.4 Branch, probability and conditional state
The returned branch keeps its complex phase. A zero-probability branch has no normalized conditional state.


In [ ]:
def branch(v,basis,outcome):
    direction = basis[:,outcome]
    raw = direction*np.vdot(direction,v)
    p = float(np.vdot(raw,raw).real)
    return raw,p,None if p == 0 else raw/np.sqrt(p)
raw,p,conditional = branch(psi,Z_basis,0)
assert np.isclose(p,5/9)
assert np.allclose(conditional,[(1+2j)/np.sqrt(5),0])
assert Statevector(conditional).equiv(Statevector(zero))
_,impossible,undefined = branch(zero,Z_basis,1)
assert impossible == 0 and undefined is None
print("Branch:",raw,"Probability:",p,"Conditional:",conditional)
def measure(v,basis,rng):
    p = probabilities(v,basis)
    outcome = int(rng.choice(2,p=p/p.sum()))
    _,prob,state = branch(v,basis,outcome)
    return outcome,prob,state
_,_,retained = measure(plus,Z_basis,rng)
retained_prob = probabilities(retained,Z_basis)
assert np.isclose(retained_prob.max(),1)
for _ in range(10):
    _,_,new = measure(retained,Z_basis,rng)
    assert Statevector(new).equiv(Statevector(retained))
# An intervening X measurement changes the state that will next be measured in Z.
for x_outcome in [0,1]:
    _,p,x_state = branch(zero,X_basis,x_outcome)
    assert np.isclose(p,.5)
    assert np.allclose(probabilities(x_state,Z_basis),[.5,.5])


## 4.5 Bloch coordinates and half-angles
The sphere represents pure states up to global phase. The coordinates are not amplitudes.


In [ ]:
theta,phi = 2*np.pi/3,np.pi/2
state = np.array([np.cos(theta/2),np.exp(1j*phi)*np.sin(theta/2)])
alpha,beta = state
bloch = np.array([2*(alpha.conjugate()*beta).real,
                  2*(alpha.conjugate()*beta).imag, abs(alpha)**2-abs(beta)**2])
assert np.allclose(bloch,[0,np.sqrt(3)/2,-.5])
assert np.allclose(probabilities(state,Z_basis),[.25,.75])
assert np.isclose(np.linalg.norm(bloch),1)
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(projection="3d")
u,v = np.meshgrid(np.linspace(0,2*np.pi,30),np.linspace(0,np.pi,15))
ax.plot_wireframe(np.cos(u)*np.sin(v),np.sin(u)*np.sin(v),np.cos(v),color="#ccd5dd",linewidth=.4)
ax.quiver(0,0,0,*bloch,color="#286a9b",linewidth=2)
ax.set(xlabel="x",ylabel="y",zlabel="z",title="Pure-state Bloch point")
ax.set_box_aspect([1,1,1])
plt.show()


## 4.6 Compare coherent and randomized preparations
For a randomized preparation, average the probability predictions of its classical branches. Do not replace that procedure with a coherent sum of amplitudes.


In [ ]:
for name,basis in [("Z",Z_basis),("X",X_basis)]:
    coherent = probabilities(plus,basis)
    randomized = (probabilities(zero,basis)+probabilities(one,basis))/2
    print(name,"pure plus:",coherent,"randomized basis:",randomized)
    assert np.allclose(randomized,[.5,.5])
assert np.allclose(probabilities(plus,X_basis),[1,0])
records = []
for _ in range(100):
    preparation = int(rng.integers(2))
    outcome,_,_ = measure([zero,one][preparation],X_basis,rng)
    records.append((preparation,outcome))
print("First five simulator preparation/outcome records:",records[:5])
print("X frequencies:",np.bincount([r[1] for r in records],minlength=2)/len(records))


## Cumulative exercises
1. For the complex worked state, calculate both Z and X probabilities by hand before running the checks.
2. Explain why multiplying the entire state by i changes neither distribution.
3. Normalize (1-i,2), then calculate the conditional outcome-0 column without prematurely discarding its phase.
4. Specify how fresh preparation differs from retained-state remeasurement.
5. Explain why one Z record does not identify the original state.
6. Compare a coherent plus preparation and an equal randomized basis preparation in both bases.

The chapter overview contains hints and worked reasoning. This notebook models ideal states and measurements; no hardware account is needed.

Programming reference: [IBM Qiskit Statevector documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector). In particular, read the distinction between sample_counts and a conditional measurement update.
